# Decoding Customer Value
Prepared by Soumabha Bhim. Executed analysis walkthrough for the supplied data.

The canonical cleaning/feature logic is `src/pipeline.py`; run it before this notebook if regenerating outputs. No timestamps or churn labels exist, so retention and customer value are explicit proxies. All outputs below come from the included customer files and SQL database.

In [1]:
from pathlib import Path
import pandas as pd, json, sqlite3
ROOT = Path.cwd()
if not (ROOT/'data').exists(): ROOT=ROOT.parent
P=ROOT/'data/processed'
d=pd.read_csv(P/'customers_features.csv')
raw=pd.read_csv(ROOT/'data/raw/Dataset.csv')
metrics=json.loads((P/'metrics.json').read_text())
print('Customer rows:',len(d),'| Original fields:',len(raw.columns),'| Engineered total:',len(d.columns))

Customer rows: 3900 | Original fields: 18 | Engineered total: 37


## 1. Audit and cleaning
Missing ratings remain Unknown. Frequency synonyms are normalized. Promo and discount flags must not be added as separate signals.

In [2]:
print(raw.isna().sum()[raw.isna().sum()>0].to_string())
print(pd.crosstab(raw.Gender,raw['Discount Applied']).to_string())
print(pd.crosstab(raw['Subscription Status'],raw['Discount Applied']).to_string())
print('Promo and discount identical:',raw['Promo Code Used'].eq(raw['Discount Applied']).all())

Review Rating    37
Discount Applied    No   Yes
Gender                      
Female            1248     0
Male               975  1677
Discount Applied       No   Yes
Subscription Status            
No                   2223   624
Yes                     0  1053
Promo and discount identical: True


## 2. Inspect engineered metrics
Dependency score is current-purchase exposure (0/100), not historical dependence. Annual spend proxy assumes stable basket and stated frequency. High/Core tiers use fixed full-sample thresholds $1,924/$572; satisfaction >=4; missing satisfaction Unknown.

In [3]:
print(d[['customer_id','annual_orders_proxy','dependency_score','annual_spend_proxy_usd','value_tier','satisfaction_flag','loyalty_a','loyalty_b','segment']].head(12).to_string(index=False))
print(pd.read_csv(P/'value_tiers.csv').to_string(index=False))

 customer_id  annual_orders_proxy  dependency_score  annual_spend_proxy_usd value_tier satisfaction_flag  loyalty_a  loyalty_b                   segment
           1                   26               100                    1378       Core  Service recovery          0          0  Developing promo exposed
           2                   26               100                    1664       Core  Service recovery          0          0  Developing promo exposed
           3                   52               100                    3796       High  Service recovery          0          0  Developing promo exposed
           4                   52               100                    4680       High           Neutral          1          0 Established promo exposed
           5                    1               100                      49 Developing  Service recovery          0          0  Developing promo exposed
           6                   52               100                    1040       

## 3. Test competing loyalty definitions
A = previous purchases >=26 and annual cadence >=12. B = previous purchases >=26 and known rating >=4. A is the primary operational repeat-behavior proxy. B shows a small independent AOV lift, while A does not. A better captures repeat cadence; neither is validated for future retention. Annual-spend differences for A are partly circular. Bootstrap intervals are exploratory, unadjusted and assume independent customers.

In [4]:
print(pd.read_csv(P/'loyalty_comparison.csv').round(4).to_string(index=False))
print('Jaccard overlap:',round(metrics['definition_jaccard'],4))
print(pd.read_csv(P/'threshold_sensitivity.csv').round(3).to_string(index=False))

definition  customers  share     aov  other_aov  aov_lift  aov_difference  aov_ci_low  aov_ci_high  annual_spend_proxy_mean  promo_rate  satisfied_rate
 loyalty_a       1086 0.2785 59.5801    59.8355   -0.0043         -0.2554     -1.9714       1.3804                1733.5083      0.4392          0.4291
 loyalty_b        817 0.2095 61.2974    59.3581    0.0327          1.9393      0.1605       3.8357                1082.3097      0.4100          1.0000
Jaccard overlap: 0.3243
 history_threshold  annual_order_threshold  customers  share  aov_usd
                20                       4       2076  0.532   59.765
                20                      12       1350  0.346   59.540
                20                      26       1004  0.257   59.932
                26                       4       1669  0.428   59.884
                26                      12       1086  0.278   59.580
                26                      26        812  0.208   59.826
                30            

## 4. SQL answers to all five questions
These outputs are descriptive. No view establishes causal discount dependency, future prediction or market underpenetration.

In [5]:
conn=sqlite3.connect(P/'customer_intelligence.sqlite')
for view in ['q1_segments','q2_behavior','q3_geography','q3_demographics','q4_promo','q5_icp']:
    print('\n'+view)
    print(pd.read_sql_query('SELECT * FROM '+view+' LIMIT 8',conn).round(3).to_string(index=False))


q1_segments
                  segment  customers  observed_sales_usd  aov_usd  promo_rate  loyalty_proxy_rate  mean_previous_purchases  satisfied_share_all  mean_known_rating
    Developing full price       1614               96732   59.933         0.0                 0.0                   20.395                0.415              3.745
 Developing promo exposed       1200               71645   59.704         1.0                 0.0                   20.853                0.415              3.747
   Established full price        609               36938   60.654         0.0                 1.0                   37.409                0.447              3.792
Established promo exposed        477               27766   58.210         1.0                 1.0                   38.044                0.407              3.720

q2_behavior
  cadence_group satisfaction_flag  customers  aov_usd  mean_previous_purchases  high_history_rate  promo_rate
         Annual           Neutral        114   56

## 5. Category / season history composition
History counts refer to the brand, not the observed category. These are not observed customer journeys or tenure.

In [6]:
for view in ['category_funnel','season_patterns']:
    print('\n'+view)
    print(pd.read_sql_query('SELECT * FROM '+view,conn).round(3).to_string(index=False))
conn.close()


category_funnel
   category        history_band  customers  aov_usd  promo_rate
Accessories  High history (26+)        627   59.263       0.442
Accessories  Low history (0-10)        242   62.632       0.413
Accessories Mid history (11-25)        371   58.989       0.447
   Clothing  High history (26+)        853   60.834       0.408
   Clothing  Low history (0-10)        348   59.664       0.408
   Clothing Mid history (11-25)        536   58.974       0.450
   Footwear  High history (26+)        297   60.384       0.444
   Footwear  Low history (0-10)        119   60.395       0.395
   Footwear Mid history (11-25)        183   59.956       0.437
  Outerwear  High history (26+)        158   56.829       0.475
  Outerwear  Low history (0-10)         75   59.427       0.373
  Outerwear Mid history (11-25)         91   55.912       0.451

season_patterns
season        history_band  customers  aov_usd  promo_rate
  Fall  High history (26+)        464   60.894       0.435
  Fall  Low hist

## 6. Margin sensitivity and pilot
74 customers meet the exact sunset trigger. Assumptions: recorded amount is net revenue, 15% discount, COGS 60% of list sales; scenarios illustrate full removal. The first live test would reduce the current discount by 5 percentage points after instrumentation. A 37/37 pilot is a feasibility test, not proof of non-inferior sales.

In [7]:
scenarios=pd.read_csv(P/'margin_scenarios.csv')
print(scenarios[(scenarios.assumed_discount==.15)&(scenarios.cogs_share_of_list==.6)].round(2).to_string(index=False))
print('Eligible customers:',int(d.sunset_eligible.sum()))

 assumed_discount  cogs_share_of_list  volume_loss  eligible_customers  observed_cohort_sales  baseline_gross_profit  scenario_sales  scenario_gross_profit  gross_profit_change  revenue_breakeven_loss  gp_breakeven_loss
             0.15                 0.6         0.00                  74                 5356.0                1575.29         6301.18                2520.47               945.18                    0.15               0.37
             0.15                 0.6         0.05                  74                 5356.0                1575.29         5986.12                2394.45               819.15                    0.15               0.37
             0.15                 0.6         0.10                  74                 5356.0                1575.29         5671.06                2268.42               693.13                    0.15               0.37
             0.15                 0.6         0.20                  74                 5356.0                1575.29    

## 7. Ideal customer profile
103-customer seed: High tier + loyalty A + satisfied + current full-price observation. Frequency is mechanically selected, and gender composition is biased by offer eligibility. Use behavior and category creative with randomized acquisition testing.

In [8]:
profile=pd.read_csv(P/'icp_profile.csv')
print(profile.groupby('field',sort=False).head(3).round(3).to_string(index=False))
print('Checks:',json.loads((P/'validation.json').read_text())['status'])

         field           level  icp_customers  icp_share  base_share  index_vs_base
      age_band           25-34             25      0.243       0.194          1.254
      age_band           45-54             22      0.214       0.193          1.108
      age_band           55-64             18      0.175       0.193          0.908
        gender          Female             56      0.544       0.320          1.699
        gender            Male             47      0.456       0.680          0.671
      category        Clothing             45      0.437       0.445          0.981
      category     Accessories             25      0.243       0.318          0.763
      category        Footwear             24      0.233       0.154          1.517
payment_method     Credit Card             25      0.243       0.172          1.411
payment_method   Bank Transfer             20      0.194       0.157          1.237
payment_method            Cash             19      0.184       0.172        

## 8. Reproduce / extend
From the project root: `python -m pip install -r requirements.txt`, then `python run_project.py`.

Open `powerbi/Founder.pbip` for the native authored report. Power BI Desktop refresh and visuals require local verification; schema validation is not a Desktop execution test. Open `dashboard/Founder_Dashboard.html` for the portable interactive companion. The PDFs provide findings, a phased plan and a one-page executive summary. Collect dates, real discounts, costs, assignment and future outcomes before attempting survival analysis, causal treatment effects or genuine retention modeling.